In [1]:
!pip install pandas numpy matplotlib seaborn tqdm scikit-learn datasets
!pip install emoji
!pip uninstall nltk -y
!pip install nltk
!pip install spacy
!pip install optuna
!python -m spacy download en_core_web_sm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which 

In [2]:
# 📌 File and Directory Management
import os
import tarfile
from google.colab import files  # Para baixar arquivos no Google Colab

# 📌 Data Manipulation
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from collections import Counter
import itertools

# 📌 Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# 📌 Text Processing (NLP)
import re
import emoji
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from textblob import TextBlob

# 📌 Machine Learning and Model Evaluation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score, confusion_matrix, roc_curve

# 📌 Deep Learning (Transformers and PyTorch)
import torch
from torch.utils.data import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# 📌 Hyperparameter Optimization
import optuna

# 📌 Utilities
from tqdm import tqdm

# 📌 Pandas Display Settings
pd.set_option("display.max_colwidth", None)  # Exibir colunas de texto completas

# 📌 NLTK Downloads (se necessário)
nltk.download('stopwords')
nltk.download('punkt')

In [3]:
# File path for the tar archive
tar_path = "aclImdb_v1.tar"

# Extracting the files
with tarfile.open(tar_path, "r") as tar:
    tar.extractall()  # Extracts to the current directory

print("File extracted successfully!")


File extracted successfully!


In [4]:
# Dataset path
dataset_path = "aclImdb"

# Function to load data
def load_imdb_data(split):
    data = []
    labels = []
    for sentiment, label in [("pos", 1), ("neg", 0)]:
        path = os.path.join(dataset_path, split, sentiment)
        for filename in tqdm(os.listdir(path), desc=f"Loading {split}/{sentiment}"):
            with open(os.path.join(path, filename), "r", encoding="utf-8") as file:
                data.append(file.read())
                labels.append(label)
    return pd.DataFrame({"review": data, "sentiment": labels})

# Loading
df_train = load_imdb_data("train")
df_test = load_imdb_data("test")

Loading test/neg: 100%|██████████| 12500/12500 [00:00<00:00, 17856.39it/s]


# Simple Training of Traditional and BERT Models

In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
print("Modelo baixado com sucesso!")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modelo baixado com sucesso!


In [6]:
# Disable Weights & Biases
os.environ["WANDB_MODE"] = "disabled"

# Load data
df_train = pd.read_csv('imdb_train_preprocessed.csv')
df_test = pd.read_csv('imdb_test_preprocessed.csv')
df_train_bert = pd.read_csv('imdb_train_bert.csv')
df_test_bert = pd.read_csv('imdb_test_bert.csv')

# Function to calculate specificity
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

# Function to calculate KS
def ks_score(y_true, y_pred_proba):
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    return max(tpr - fpr)

# Main function to run models and compute metrics
def evaluate_models(train_data, test_data, train_labels, test_labels, is_bert=False):
    results = {}

    if not is_bert:
        # TF-IDF vectorization for traditional models
        tfidf = TfidfVectorizer(max_features=5000)
        X_train = tfidf.fit_transform(train_data)
        X_test = tfidf.transform(test_data)
        y_train = train_labels
        y_test = test_labels

        # Traditional models
        models = {
            "Logistic Regression": LogisticRegression(max_iter=1000),
            "Naive Bayes": MultinomialNB(),
            "SVM": LinearSVC(max_iter=1000)
        }

        for name, model in models.items():
            # Train
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)

            # Compute metrics
            results[name] = {
                "Accuracy": accuracy_score(y_test, y_pred),
                "F1-Score": f1_score(y_test, y_pred),
                "AUC": roc_auc_score(y_test, y_pred_proba),
                "KS": ks_score(y_test, y_pred_proba),
                "Recall": recall_score(y_test, y_pred),
                "Specificity": specificity_score(y_test, y_pred),
                "Precision": precision_score(y_test, y_pred)
            }

    else:
        # BERT
        class IMDbDataset(Dataset):
            def __init__(self, reviews, labels, tokenizer, max_length=128):
                self.encodings = tokenizer(reviews.tolist(), truncation=True, padding=True, max_length=max_length)
                self.labels = labels.tolist()

            def __getitem__(self, idx):
                item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
                item['labels'] = torch.tensor(self.labels[idx])
                return item

            def __len__(self):
                return len(self.labels)

        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
        train_dataset = IMDbDataset(train_data, train_labels, tokenizer)
        test_dataset = IMDbDataset(test_data, test_labels, tokenizer)

        training_args = TrainingArguments(
            output_dir='./results',
            num_train_epochs=3,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            warmup_steps=500,
            weight_decay=0.01,
            logging_dir='./logs',
            eval_strategy="epoch",  # Updated evaluation_strategy (deprecated)
            report_to="none"  # Disables external reporting like W&B
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
        )

        trainer.train()
        predictions = trainer.predict(test_dataset)
        y_pred = predictions.predictions.argmax(-1)
        y_pred_proba = torch.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()

        results["BERT"] = {
            "Accuracy": accuracy_score(test_labels, y_pred),
            "F1-Score": f1_score(test_labels, y_pred),
            "AUC": roc_auc_score(test_labels, y_pred_proba),
            "KS": ks_score(test_labels, y_pred_proba),
            "Recall": recall_score(test_labels, y_pred),
            "Specificity": specificity_score(test_labels, y_pred),
            "Precision": precision_score(test_labels, y_pred)
        }

    return results

# Run traditional models
results_traditional = evaluate_models(
    df_train['clean_review'], df_test['clean_review'],
    df_train['sentiment'], df_test['sentiment'],
    is_bert=False
)

# Run BERT
results_bert = evaluate_models(
    df_train_bert['clean_review'], df_test_bert['clean_review'],
    df_train_bert['sentiment'], df_test_bert['sentiment'],
    is_bert=True
)

# Combine results
all_results = {**results_traditional, **results_bert}

# Display results as a table
results_df = pd.DataFrame(all_results).T
print("\nModel Results:")
print(results_df.round(4))

# Save results
results_df.to_csv('model_comparison_results.csv')
print("📌 Results saved to 'model_comparison_results.csv'")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.336300,0.354996
2,0.268600,0.456123
3,0.106000,0.600034



Resultados dos Modelos:
                     Acurácia  F1-Score     AUC      KS  \
Logistic Regression    0.8798    0.8808  0.9506  0.7615   
Naive Bayes            0.8418    0.8399  0.9203  0.6890   
SVM                    0.8638    0.8638  0.9394  0.7305   
BERT                   0.8866    0.8886  0.9556  0.7742   

                     Sensibilidade (Recall)  Especificidade  Precisão  
Logistic Regression                  0.8851          0.8745    0.8765  
Naive Bayes                          0.8273          0.8563    0.8528  
SVM                                  0.8614          0.8662    0.8663  
BERT                                 0.9019          0.8711    0.8757  
📌 Resultados salvos em 'model_comparison_results.csv'


# Including Metrics in Training and Adjustments in BERT

In [8]:
# Check if GPU is available
print("GPU available:" if torch.cuda.is_available() else "GPU not available")
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print(f"Allocated GPU memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Reserved GPU memory: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

# Disable Weights & Biases
os.environ["WANDB_MODE"] = "disabled"

# Load data
df_train = pd.read_csv('imdb_train_preprocessed.csv')
df_test = pd.read_csv('imdb_test_preprocessed.csv')
df_train_bert = pd.read_csv('imdb_train_bert.csv')
df_test_bert = pd.read_csv('imdb_test_bert.csv')

# Specificity score function
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

# KS score function
def ks_score(y_true, y_pred_proba):
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    return max(tpr - fpr)

# Metrics calculation function
def calculate_metrics(y_true, y_pred, y_pred_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1-Score": f1_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_pred_proba),
        "KS": ks_score(y_true, y_pred_proba),
        "Recall": recall_score(y_true, y_pred),
        "Specificity": specificity_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred)
    }

# Main function to run models and calculate metrics
def evaluate_models(train_data, test_data, train_labels, test_labels, is_bert=False):
    results_train = {}
    results_test = {}

    if not is_bert:
        # TF-IDF vectorization for traditional models
        tfidf = TfidfVectorizer(max_features=5000)
        X_train = tfidf.fit_transform(train_data)
        X_test = tfidf.transform(test_data)
        y_train = train_labels
        y_test = test_labels

        # Traditional models
        models = {
            "Logistic Regression": LogisticRegression(max_iter=1000),
            "Naive Bayes": MultinomialNB(),
            "SVM": LinearSVC(max_iter=1000)
        }

        for name, model in models.items():
            # Train
            model.fit(X_train, y_train)

            # Predictions on training set
            y_pred_train = model.predict(X_train)
            y_pred_proba_train = model.predict_proba(X_train)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_train)
            results_train[name] = calculate_metrics(y_train, y_pred_train, y_pred_proba_train)

            # Predictions on test set
            y_pred_test = model.predict(X_test)
            y_pred_proba_test = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
            results_test[name] = calculate_metrics(y_test, y_pred_test, y_pred_proba_test)

    else:
        # BERT model
        class IMDbDataset(Dataset):
            def __init__(self, reviews, labels, tokenizer, max_length=64):
                self.encodings = tokenizer(reviews.tolist(), truncation=True, padding=True, max_length=max_length)
                self.labels = labels.tolist()

            def __getitem__(self, idx):
                item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
                item['labels'] = torch.tensor(self.labels[idx])
                return item

            def __len__(self):
                return len(self.labels)

        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

        # Move model to GPU if available
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        print(f"BERT model moved to: {device}")

        train_dataset = IMDbDataset(train_data, train_labels, tokenizer)
        test_dataset = IMDbDataset(test_data, test_labels, tokenizer)

        training_args = TrainingArguments(
            output_dir='./results',
            num_train_epochs=2,
            per_device_train_batch_size=32,
            per_device_eval_batch_size=32,
            warmup_steps=500,
            weight_decay=0.1,
            logging_dir='./logs',
            eval_strategy="steps",
            eval_steps=200,
            save_strategy="steps",
            save_steps=200,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            save_total_limit=1,
            fp16=True if torch.cuda.is_available() else False
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
        )

        trainer.train()

        # Predictions on training set
        predictions_train = trainer.predict(train_dataset)
        y_pred_train = predictions_train.predictions.argmax(-1)
        y_pred_proba_train = torch.softmax(torch.tensor(predictions_train.predictions), dim=1)[:, 1].numpy()
        results_train["BERT"] = calculate_metrics(train_labels, y_pred_train, y_pred_proba_train)

        # Predictions on test set
        predictions_test = trainer.predict(test_dataset)
        y_pred_test = predictions_test.predictions.argmax(-1)
        y_pred_proba_test = torch.softmax(torch.tensor(predictions_test.predictions), dim=1)[:, 1].numpy()
        results_test["BERT"] = calculate_metrics(test_labels, y_pred_test, y_pred_proba_test)

    return results_train, results_test

# Run traditional models
results_train_traditional, results_test_traditional = evaluate_models(
    df_train['clean_review'], df_test['clean_review'],
    df_train['sentiment'], df_test['sentiment'],
    is_bert=False
)

# Run BERT
results_train_bert, results_test_bert = evaluate_models(
    df_train_bert['clean_review'], df_test_bert['clean_review'],
    df_train_bert['sentiment'], df_test_bert['sentiment'],
    is_bert=True
)

# Combine results
all_results_train = {**results_train_traditional, **results_train_bert}
all_results_test = {**results_test_traditional, **results_test_bert}

# Display results in tables
results_train_df = pd.DataFrame(all_results_train).T
results_test_df = pd.DataFrame(all_results_test).T

print("\nModel Results (Training):")
print(results_train_df.round(4))

print("\nModel Results (Test):")
print(results_test_df.round(4))

# Save results
results_train_df.to_csv('model_comparison_results_train1.csv')
results_test_df.to_csv('model_comparison_results_test1.csv')
print("📌 Training results saved to 'model_comparison_results_train1.csv'")
print("📌 Test results saved to 'model_comparison_results_test1.csv'")

# Download files in Colab
from google.colab import files
files.download('model_comparison_results_train1.csv')
files.download('model_comparison_results_test1.csv')


GPU disponível:
Nome da GPU: Tesla T4
Memória GPU alocada: 0.43 GB
Memória GPU reservada: 0.53 GB


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modelo BERT movido para: cuda


Step,Training Loss,Validation Loss
200,No log,0.431362
400,No log,0.435625
600,0.500600,0.406597
800,0.500600,0.509535
1000,0.340700,0.399049
1200,0.340700,0.373285
1400,0.340700,0.353225



Resultados dos Modelos (Treino):
                     Acurácia  F1-Score     AUC      KS  \
Logistic Regression    0.9135    0.9144  0.9718  0.8287   
Naive Bayes            0.8657    0.8664  0.9403  0.7323   
SVM                    0.9424    0.9428  0.9862  0.8871   
BERT                   0.9562    0.9562  0.9884  0.9134   

                     Sensibilidade (Recall)  Especificidade  Precisão  
Logistic Regression                  0.9229          0.9040    0.9061  
Naive Bayes                          0.8696          0.8618    0.8633  
SVM                                  0.9476          0.9372    0.9380  
BERT                                 0.9550          0.9574    0.9574  

Resultados dos Modelos (Teste):
                     Acurácia  F1-Score     AUC      KS  \
Logistic Regression    0.8798    0.8808  0.9506  0.7615   
Naive Bayes            0.8418    0.8399  0.9203  0.6890   
SVM                    0.8638    0.8638  0.9394  0.7305   
BERT                   0.8444    0.8441  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Hyperparameter Optimization with Optuna

In [12]:
# Verificar se a GPU está disponível
print("GPU disponível:" if torch.cuda.is_available() else "GPU não disponível")
if torch.cuda.is_available():
    print("Nome da GPU:", torch.cuda.get_device_name(0))
    print(f"Memória GPU alocada: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Memória GPU reservada: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

# Desativar Weights & Biases
os.environ["WANDB_MODE"] = "disabled"

# Carregar dados
df_train = pd.read_csv('imdb_train_preprocessed.csv')
df_test = pd.read_csv('imdb_test_preprocessed.csv')
df_train_bert = pd.read_csv('imdb_train_bert.csv')
df_test_bert = pd.read_csv('imdb_test_bert.csv')

# Função para calcular especificidade
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

# Função para calcular KS
def ks_score(y_true, y_pred_proba):
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    return max(tpr - fpr)

# Função para calcular métricas
def calculate_metrics(y_true, y_pred, y_pred_proba):
    return {
        "Acurácia": accuracy_score(y_true, y_pred),
        "F1-Score": f1_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_pred_proba),
        "KS": ks_score(y_true, y_pred_proba),
        "Sensibilidade (Recall)": recall_score(y_true, y_pred),
        "Especificidade": specificity_score(y_true, y_pred),
        "Precisão": precision_score(y_true, y_pred)
    }

# Dataset para o BERT
class IMDbDataset(Dataset):
    def __init__(self, reviews, labels, tokenizer, max_length=64):
        self.encodings = tokenizer(reviews.tolist(), truncation=True, padding=True, max_length=max_length)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Função de objetivo para o Optuna
def objective(trial):
    # Definir hiperparâmetros a serem otimizados
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.01, 0.1)
    num_train_epochs = trial.suggest_int("num_train_epochs", 1, 3)
    per_device_train_batch_size = trial.suggest_categorical("per_device_train_batch_size", [16, 32])
    classifier_dropout = trial.suggest_float("classifier_dropout", 0.1, 0.3)
    warmup_steps = trial.suggest_int("warmup_steps", 0, 500)

    # Configurar o modelo e o tokenizer
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2, classifier_dropout=classifier_dropout)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    train_dataset = IMDbDataset(df_train_bert['clean_review'], df_train_bert['sentiment'], tokenizer)
    test_dataset = IMDbDataset(df_test_bert['clean_review'], df_test_bert['sentiment'], tokenizer)

    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_train_batch_size,
        warmup_steps=warmup_steps,
        weight_decay=weight_decay,
        learning_rate=learning_rate,
        logging_dir='./logs',
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=1,
        fp16=True if torch.cuda.is_available() else False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
    )

    trainer.train()

    # Obter o validation loss
    eval_results = trainer.evaluate()
    return eval_results["eval_loss"]

# Função para treinar e avaliar os modelos
def evaluate_models(train_data, test_data, train_labels, test_labels, is_bert=False, best_params=None):
    results_train = {}
    results_test = {}

    if not is_bert:
        # Vetorização TF-IDF para modelos tradicionais
        tfidf = TfidfVectorizer(max_features=5000)
        X_train = tfidf.fit_transform(train_data)
        X_test = tfidf.transform(test_data)
        y_train = train_labels
        y_test = test_labels

        # Modelos tradicionais
        models = {
            "Logistic Regression": LogisticRegression(max_iter=1000),
            "Naive Bayes": MultinomialNB(),
            "SVM": LinearSVC(max_iter=1000)
        }

        for name, model in models.items():
            # Treinar
            model.fit(X_train, y_train)

            # Previsões no conjunto de treino
            y_pred_train = model.predict(X_train)
            y_pred_proba_train = model.predict_proba(X_train)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_train)
            results_train[name] = calculate_metrics(y_train, y_pred_train, y_pred_proba_train)

            # Previsões no conjunto de teste
            y_pred_test = model.predict(X_test)
            y_pred_proba_test = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
            results_test[name] = calculate_metrics(y_test, y_pred_test, y_pred_proba_test)

    else:
        # BERT com os melhores hiperparâmetros
        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2, classifier_dropout=best_params["classifier_dropout"])

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        print(f"Modelo BERT movido para: {device}")

        train_dataset = IMDbDataset(train_data, train_labels, tokenizer)
        test_dataset = IMDbDataset(test_data, test_labels, tokenizer)

        training_args = TrainingArguments(
            output_dir='./results',
            num_train_epochs=best_params["num_train_epochs"],
            per_device_train_batch_size=best_params["per_device_train_batch_size"],
            per_device_eval_batch_size=best_params["per_device_train_batch_size"],
            warmup_steps=best_params["warmup_steps"],
            weight_decay=best_params["weight_decay"],
            learning_rate=best_params["learning_rate"],
            logging_dir='./logs',
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            save_total_limit=1,
            fp16=True if torch.cuda.is_available() else False
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
        )

        trainer.train()

        # Previsões no conjunto de treino
        predictions_train = trainer.predict(train_dataset)
        y_pred_train = predictions_train.predictions.argmax(-1)
        y_pred_proba_train = torch.softmax(torch.tensor(predictions_train.predictions), dim=1)[:, 1].numpy()
        results_train["BERT"] = calculate_metrics(train_labels, y_pred_train, y_pred_proba_train)

        # Previsões no conjunto de teste
        predictions_test = trainer.predict(test_dataset)
        y_pred_test = predictions_test.predictions.argmax(-1)
        y_pred_proba_test = torch.softmax(torch.tensor(predictions_test.predictions), dim=1)[:, 1].numpy()
        results_test["BERT"] = calculate_metrics(test_labels, y_pred_test, y_pred_proba_test)

    return results_train, results_test

# Realizar a otimização bayesiana com Optuna
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)  # Testar 10 combinações de hiperparâmetros

# Exibir os melhores hiperparâmetros
print("Melhores hiperparâmetros encontrados:")
print(study.best_params)
print(f"Melhor validation loss: {study.best_value}")

# Treinar os modelos tradicionais
results_train_traditional, results_test_traditional = evaluate_models(
    df_train['clean_review'], df_test['clean_review'],
    df_train['sentiment'], df_test['sentiment'],
    is_bert=False
)

# Treinar o BERT com os melhores hiperparâmetros
results_train_bert, results_test_bert = evaluate_models(
    df_train_bert['clean_review'], df_test_bert['clean_review'],
    df_train_bert['sentiment'], df_test_bert['sentiment'],
    is_bert=True,
    best_params=study.best_params
)

# Combinar resultados
all_results_train = {**results_train_traditional, **results_train_bert}
all_results_test = {**results_test_traditional, **results_test_bert}

# Exibir resultados em tabelas
results_train_df = pd.DataFrame(all_results_train).T
results_test_df = pd.DataFrame(all_results_test).T

print("\nResultados dos Modelos (Treino):")
print(results_train_df.round(4))

print("\nResultados dos Modelos (Teste):")
print(results_test_df.round(4))

# Salvar resultados
results_train_df.to_csv('model_comparison_results_train3.csv')
results_test_df.to_csv('model_comparison_results_test3.csv')
print("📌 Resultados de treino salvos em 'model_comparison_results_train3.csv'")
print("📌 Resultados de teste salvos em 'model_comparison_results_test3.csv'")

# Baixar os arquivos no Colab
from google.colab import files
files.download('model_comparison_results_train3.csv')
files.download('model_comparison_results_test3.csv')

GPU disponível:
Nome da GPU: Tesla T4
Memória GPU alocada: 1.68 GB
Memória GPU reservada: 2.95 GB


[I 2025-03-22 21:18:37,821] A new study created in memory with name: no-name-c008529b-dbfa-4a09-a775-9e64ef0bb941
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.452500,0.359240
2,0.257600,0.383363
3,0.158700,0.477850


[I 2025-03-22 21:29:17,587] Trial 0 finished with value: 0.3592395484447479 and parameters: {'learning_rate': 2.6207047611330844e-05, 'weight_decay': 0.0842411953680113, 'num_train_epochs': 3, 'per_device_train_batch_size': 32, 'classifier_dropout': 0.24109558790640712, 'warmup_steps': 11}. Best is trial 0 with value: 0.3592395484447479.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.384700,0.349664
2,0.224600,0.405026


...


Resultados dos Modelos (Treino):
                     Acurácia  F1-Score     AUC      KS  \
Logistic Regression    0.9135    0.9144  0.9718  0.8287   
Naive Bayes            0.8657    0.8664  0.9403  0.7323   
SVM                    0.9424    0.9428  0.9862  0.8871   
BERT                   0.9154    0.9157  0.9717  0.8311   

                     Sensibilidade (Recall)  Especificidade  Precisão  
Logistic Regression                  0.9229          0.9040    0.9061  
Naive Bayes                          0.8696          0.8618    0.8633  
SVM                                  0.9476          0.9372    0.9380  
BERT                                 0.9176          0.9132    0.9138  

Resultados dos Modelos (Teste):
                     Acurácia  F1-Score     AUC      KS  \
Logistic Regression    0.8798    0.8808  0.9506  0.7615   
Naive Bayes            0.8418    0.8399  0.9203  0.6890   
SVM                    0.8638    0.8638  0.9394  0.7305   
BERT                   0.8428    0.8435  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>